# 🚀 Fine-Tuning Open-Source LLM (Google Gemma 2B) for UTHM Academic Chatbot
**Project:** UTHM Academic AI Assistant (PSM/FYP)
**Method:** QLoRA 4-bit Quantization + SFTTrainer
**GPU Required:** NVIDIA T4 (Free Google Colab GPU)

In [ ]:
# 1. Install required high-performance LLM libraries
!pip install -q -U torch transformers datasets peft accelerate bitsandbytes trl unsloth

In [ ]:
# 2. Load UTHM Instruction Dataset
import pandas as pd
from datasets import Dataset

df = pd.read_csv('uthm_instruction_dataset.csv')
dataset = Dataset.from_pandas(df)
print(f"Loaded {len(dataset)} instruction pairs for fine-tuning.")

In [ ]:
# 3. Format Prompt Template
def format_prompt(example):
    return f"""### System Instruction:
{example['instruction']}

### User Query:
{example['input']}

### UTHM Assistant Response:
{example['output']}"""

dataset = dataset.map(lambda x: {"text": format_prompt(x)})

In [ ]:
# 4. Load Base Model (Gemma-2b-it) in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 5. Add LoRA Adapters for Parameter Efficient Fine-Tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
# 6. Train Model with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 7. Test Fine-Tuned Model Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    format_prompt({
        "instruction": "Anda adalah Pembantu AI Akademik Rasmi UTHM.",
        "input": "mcm mana nk dpt 2nd class upper uthm?",
        "output": ""
    })
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# 8. Save Fine-Tuned Model to Google Drive
model.save_pretrained("fine_tuned_gemma_uthm")
tokenizer.save_pretrained("fine_tuned_gemma_uthm")
print("✅ Model Fine-Tuning Complete & Saved Successfully!")